Hybrid Fraud Detection: Node2Vec + XGBoost/Balanced RF
This notebook implements a graph-enhanced fraud detection system. We represent transactions as a directed graph, learn account embeddings using Node2Vec, and combine them with tabular features for classification.

Dataset: PaySim (Realistic Financial Simulator).

Dependencies

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch_geometric.nn.models import Node2Vec
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from imblearn.ensemble import BalancedRandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report

# Device configuration for GPU acceleration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

Data Loading & Filtering

In [ ]:
os.environ['KAGGLE_USERNAME'] = "your_username"
os.environ['KAGGLE_KEY'] = "your_key"

# Download dataset
!kaggle datasets download -d ealaxi/paysim1
!unzip -o paysim1.zip

# Load dataset
df = pd.read_csv("paysim.csv")

# Filter relevant transactions
df = df[(df['type'] == 'CASH_OUT') | (df['type'] == 'TRANSFER')].copy()

# One-hot encode
df = pd.get_dummies(df, columns=['type'], drop_first=False)

Graph Construction

In [ ]:
# Encode accounts as graph nodes
senders = df['nameOrig'].astype('category')
receivers = df['nameDest'].astype('category')
all_nodes = pd.Index(senders.cat.categories).union(pd.Index(receivers.cat.categories))
node_map = pd.Series(np.arange(len(all_nodes)), index=all_nodes)

src = node_map.loc[senders.astype(str)].to_numpy()
dst = node_map.loc[receivers.astype(str)].to_numpy()
edge_index = torch.tensor(np.vstack([src, dst]), dtype=torch.long).to(device)

Node2Vec Training

In [ ]:
# Initialize Node2Vec Model
model = Node2Vec(
    edge_index,
    embedding_dim=32,
    walk_length=10,
    context_size=5,
    walks_per_node=2,
    num_negative_samples=2,
    sparse=True
).to(device)

# Extract learned node embeddings
model.eval()
with torch.no_grad():
    emb = model().cpu().numpy()

Hybrid Feature Engineering

In [ ]:
# Select tabular transaction features
tab_cols = ['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest'] + \
           [c for c in df.columns if c.startswith('type_')]

X_tab = df[tab_cols].fillna(0).to_numpy()
X_src, X_dst = emb[src], emb[dst]

# Hybrid feature set: (Source Embedding + Destination Embedding + Tabular)
X = np.hstack([X_src, X_dst, X_tab]).astype(np.float32)
y = df['isFraud'].astype(int).to_numpy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = RobustScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

Hybrid Model 1 – Balanced Random Forest

This implements the version that manages imbalance using weighted sampling instead of SMOTE.

In [ ]:
# Train Balanced Random Forest (Hybrid-1)
brf = BalancedRandomForestClassifier(n_estimators=800, n_jobs=-1, random_state=42)
brf.fit(X_train_s, y_train)

# Evaluation at optimized threshold
probs_brf = brf.predict_proba(X_test_s)[:, 1]
best_t = 0.76
preds_brf = (probs_brf >= best_t).astype(int)

print("\n=== HYBRID MODEL 1: Node2Vec + Balanced Random Forest ===")
print("HYBRID AUROC:", roc_auc_score(y_test, probs_brf))
print("HYBRID AUPRC:", average_precision_score(y_test, probs_brf))
print(classification_report(y_test, preds_brf))

Hybrid Model 2 – XGBoost
The final proposed model with optimized hyperparameters.

In [ ]:
# Train XGBoost (Hybrid-2)
xgb = XGBClassifier(n_estimators=600, max_depth=6, learning_rate=0.05, subsample=0.9,
                    colsample_bytree=0.9, scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
                    eval_metric='logloss', random_state=42)

xgb.fit(X_train_s, y_train)

# Evaluation
probs_xgb = xgb.predict_proba(X_test_s)[:, 1]
preds_xgb = (probs_xgb >= best_t).astype(int)

print("\n=== HYBRID MODEL 2: Node2Vec + XGBoost ===")
print("HYBRID AUROC:", roc_auc_score(y_test, probs_xgb))
print("HYBRID AUPRC:", average_precision_score(y_test, probs_xgb))
print(classification_report(y_test, preds_xgb))
